# 108. Document Understanding: Processing Documents

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/amerob/ultimate-prompt-engineering-playbook/blob/main/notebooks/13-multi-modal/108_document_understanding.ipynb)

**Category:** 13 - Multi-Modal Techniques  
**Technique #:** 108  
**Difficulty:** Advanced

## 📖 Description

Document Understanding combines OCR, layout analysis, and natural language processing to extract structured information from documents. This technique handles invoices, forms, contracts, and multi-page documents with complex layouts.

### When to Use:
- Processing invoices and receipts
- Extracting data from forms
- Analyzing contracts and legal documents
- Digitizing paper documents
- Automated document classification

## 🔧 How It Works

```
┌─────────────────────────────────────────────────────────────┐
│                DOCUMENT UNDERSTANDING FLOW                   │
└─────────────────────────────────────────────────────────────┘

    ┌──────────────┐         ┌──────────────┐         ┌──────────────┐
    │   Document   │────────▶│   Layout     │────────▶│   Content    │
    │   Image      │         │   Analysis   │         │   Regions    │
    └──────────────┘         └──────────────┘         └──────┬───────┘
                                                             │
                                    ┌────────────────────────┼────────────────────────┐
                                    ▼                        ▼                        ▼
                            ┌──────────────┐         ┌──────────────┐         ┌──────────────┐
                            │   Text       │         │   Table      │         │   Form       │
                            │   Extraction │         │   Parsing    │         │   Fields     │
                            └──────────────┘         └──────────────┘         └──────────────┘
                                    │                        │                        │
                                    └────────────────────────┼────────────────────────┘
                                                             ▼
                                                      ┌──────────────┐
                                                      │  Structured  │
                                                      │   Output     │
                                                      └──────────────┘
```

### Document Types:
- **Structured**: Forms, invoices, receipts
- **Semi-structured**: Resumes, certificates
- **Unstructured**: Letters, articles, contracts

## 🛠️ Setup

In [ ]:
!pip install -q openai pillow requests

In [ ]:
import os
from getpass import getpass
import base64
import requests
import json

api_key = getpass("Enter your OpenAI API key: ")
os.environ["OPENAI_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI()

## 💡 Basic Example

In [ ]:
def encode_image(image_source):
    """Encode image to base64."""
    if image_source.startswith(('http://', 'https://')):
        response = requests.get(image_source)
        return base64.b64encode(response.content).decode('utf-8')
    with open(image_source, "rb") as f:
        return base64.b64encode(f.read()).decode('utf-8')

def understand_document(image_source, doc_type="general", model="gpt-4o"):
    """Extract information from a document image."""
    
    doc_prompts = {
        "general": """
        Analyze this document and provide:
        1. Document type
        2. Key information extracted
        3. Main sections identified
        4. Important dates, names, or numbers
        """,
        "invoice": """
        Extract invoice information as JSON:
        {
          "invoice_number": "",
          "date": "",
          "vendor_name": "",
          "vendor_address": "",
          "customer_name": "",
          "line_items": [{"description": "", "quantity": 0, "unit_price": 0, "total": 0}],
          "subtotal": 0,
          "tax": 0,
          "total_amount": 0,
          "payment_terms": ""
        }
        """,
        "resume": """
        Extract resume information as JSON:
        {
          "name": "",
          "contact": {"email": "", "phone": "", "location": ""},
          "summary": "",
          "experience": [{"company": "", "title": "", "dates": "", "description": ""}],
          "education": [{"institution": "", "degree": "", "dates": ""}],
          "skills": []
        }
        """
    }
    
    prompt = doc_prompts.get(doc_type, doc_prompts["general"])
    base64_image = encode_image(image_source)
    
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=2000
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Example: Invoice processing
invoice_url = "https://templates.mediamodifier.com/645124ff36ed6d3a3f3e1c46/supermarket-receipt-template.jpg"

print("DOCUMENT UNDERSTANDING - INVOICE EXAMPLE\n")
print("="*60 + "\n")

invoice_data = understand_document(invoice_url, doc_type="invoice")
print(invoice_data)

## 🌍 Real-World Example

In [ ]:
# Real-world: Contract analysis
def analyze_contract(image_source):
    """Comprehensive contract analysis."""
    
    prompt = """
    You are a legal document analyzer. Review this contract and provide:
    
    ## Document Overview
    - Contract type
    - Parties involved
    - Effective date and term
    
    ## Key Terms
    - Payment terms
    - Deliverables
    - Termination conditions
    
    ## Risk Assessment
    - Unusual clauses
    - Missing standard provisions
    - Potential concerns
    
    ## Action Items
    - Required signatures
    - Deadlines
    - Follow-up needed
    """
    
    base64_image = encode_image(image_source)
    
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=1500
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Error: {str(e)}"

# Example contract image
contract_url = "https://images.template.net/wp-content/uploads/2015/04/Simple-Contract-Template.jpg"

print("CONTRACT ANALYSIS\n")
print("="*60 + "\n")

contract_analysis = analyze_contract(contract_url)
print(contract_analysis)

## ❌ Failure Case

In [ ]:
# Failure case: Poor quality scans and complex layouts
print("DOCUMENT UNDERSTANDING LIMITATIONS\n")
print("="*60 + "\n")

limitations = [
    {
        "issue": "Poor scan quality",
        "impact": "Text recognition errors, missing content",
        "solution": "Use 300+ DPI, enhance contrast before processing"
    },
    {
        "issue": "Handwritten content",
        "impact": "Lower accuracy, especially for cursive",
        "solution": "Use specialized handwriting recognition tools"
    },
    {
        "issue": "Multi-column layouts",
        "impact": "Reading order confusion",
        "solution": "Specify reading order in prompt"
    },
    {
        "issue": "Tables without borders",
        "impact": "Table structure not recognized",
        "solution": "Request explicit table parsing"
    }
]

for lim in limitations:
    print(f"⚠️  {lim['issue']}")
    print(f"   Impact: {lim['impact']}")
    print(f"   Solution: {lim['solution']}\n")

print("="*60)
print("NOTE: Always verify critical data extracted from documents.")

## 📊 Benchmark Comparison

| Document Type | GPT-4o | Claude 3.5 | Azure DI | Google DAI |
|---------------|--------|------------|----------|------------|
| Invoices | 92% | 90% | 96% | 94% |
| Receipts | 89% | 87% | 93% | 91% |
| Forms | 85% | 83% | 91% | 89% |
| Contracts | 78% | 76% | 82% | 80% |
| Resumes | 88% | 86% | 90% | 88% |

*Accuracy = Field-level extraction accuracy

### When to Use Each:
- **Vision LLMs**: Complex understanding, reasoning tasks
- **Azure Document Intelligence**: Production, high-volume
- **Google Document AI**: Integration with GCP ecosystem

## 🎮 Interactive Playground

In [ ]:
def document_playground():
    """Interactive document understanding playground."""
    print("\n" + "="*60)
    print("DOCUMENT UNDERSTANDING PLAYGROUND")
    print("="*60 + "\n")
    
    image_url = input("Enter document image URL (or press Enter for sample): ").strip()
    if not image_url:
        image_url = "https://templates.mediamodifier.com/645124ff36ed6d3a3f3e1c46/supermarket-receipt-template.jpg"
    
    print("\nSelect document type:")
    print("1. General document")
    print("2. Invoice/Receipt")
    print("3. Resume/CV")
    print("4. Custom analysis")
    
    doc_choice = input("Enter choice (1-4): ").strip()
    
    doc_types = {
        "1": "general",
        "2": "invoice",
        "3": "resume",
        "4": None
    }
    
    selected_type = doc_types.get(doc_choice, "general")
    
    if doc_choice == "4" or selected_type is None:
        custom_prompt = input("\nEnter your custom document analysis request: ")
        base64_image = encode_image(image_url)
        response = client.chat.completions.create(
            model="gpt-4o",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": custom_prompt},
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ],
            max_tokens=1500
        )
        result = response.choices[0].message.content
    else:
        result = understand_document(image_url, doc_type=selected_type)
    
    print("\n" + "="*60)
    print("EXTRACTED INFORMATION:")
    print("="*60)
    print(result)

document_playground()

## 💡 Tips & Tricks

### Best Practices:
1. **Specify output format**: JSON for structured data extraction
2. **Define field types**: "date", "currency", "percentage"
3. **Request confidence scores**: For critical fields
4. **Handle missing data**: Specify null/empty handling

### Document-Specific Tips:
- **Invoices**: Ask for line-item breakdown
- **Forms**: Request field-by-field extraction
- **Contracts**: Ask for clause categorization
- **Resumes**: Request skills normalization

### Quality Improvement:
- Pre-process: deskew, denoise, enhance
- Use 300+ DPI for scans
- Crop to relevant regions
- Process multi-page documents separately

## 📚 References

1. [Azure Document Intelligence](https://azure.microsoft.com/en-us/services/form-recognizer/)
2. [Google Document AI](https://cloud.google.com/document-ai)
3. [AWS Textract](https://aws.amazon.com/textract/)
4. [LayoutLM: Pre-training of Text and Layout](https://arxiv.org/abs/1912.13318)